In [1]:
%who

Interactive namespace is empty.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Loading the data from file
df = pd.read_csv(r"C:\Users\user\Documents\00. Github Projects\01. Project 1 Credit-Risk Preditction\01. Data\accepted_2007_to_2018q4.csv\Loans Given 2007-2018 Q4.csv", low_memory=False)
print(f"Loaded: {len(df)} loans")

In [21]:
# Columns to be dropped
cols = [
    # Category 4: Payment History (Data Leakage) - drop these
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high',
    'last_fico_range_low', 'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_amount',
    'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount', 'hardship_last_payment_amount', 'debt_settlement_flag',
    'debt_settlement_flag_date', 'settlement_status', 'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term',
    'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 
    # Category 5: Internal/System (Useless) - drop these
    'url', 'desc', 'title', 'zip_code', 'addr_state', 'initial_list_status', 'policy_code', 'application_type', 'disbursement_method', 
    'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high',
    'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il',
    'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog',
    'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m',
    'max_bal_bc', 'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy',
    'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
    'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd',
    'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats',
    'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
    'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit', 'total_il_high_credit_limit', 
    # Extra Columns to be dropped
    'emp_title', 'member_id', 'emp_length'
]

# Drop all Category 4 and 5 columns
df_clean = df.drop(columns=cols, errors='ignore')

print(f"Columns remaining: {df_clean.shape[1]}")

Columns remaining: 31


In [22]:
# Filling delinquency/record columns with 0 (meaning "no event")
zero_fill = ['mths_since_last_record', 'mths_since_last_major_derog', 'mths_since_last_delinq']
for col in zero_fill:
    df_clean[col] = df_clean[col].fillna(0)

# Replacing empty values with MEDIAN (for these large-missing columns)
median_fill = ['mort_acc', 'revol_util', 'dti']
for col in median_fill:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Removing rows where the remaining columns have missing values (the ones at the end with a handful of missing values)
df_clean = df_clean.dropna()

print(f"Final rows: {len(df_clean)}")
print(f"Columns remaining: {df_clean.shape}")

Final rows: 2260638
Columns remaining: (2260638, 31)


In [23]:
# 1. CREATE TARGET
bad_statuses = ['Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off']
df_clean['is_bad'] = np.where(df_clean['loan_status'].isin(bad_statuses), 1, 0)

# 2. REMOVE ACTIVE LOANS
df_clean = df_clean[~df_clean['loan_status'].isin(['Current', 'In Grace Period'])]

print(f"Rows after removing active loans: {len(df_clean)}")
print(f"Default rate: {df_clean['is_bad'].mean()*100:.2f}%")

# 3. DROP loan_status (no longer needed)
df_clean = df_clean.drop(columns=['loan_status'], errors='ignore')

# 4. SHOW FINAL COLUMNS
print(f"Final columns ({df_clean.shape[1]}):")
print(df_clean.columns.tolist())

# 5. SAVE
df_clean.to_csv(r"C:\Users\user\Documents\00. Github Projects\01. Project 1 Credit-Risk Preditction\02. Clean Data\clean_loans_final.csv", index=False)
print("Saved to: clean_loans_final.csv")

Rows after removing active loans: 1373885
Default rate: 19.61%
Final columns (31):
['id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'pymnt_plan', 'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mths_since_last_major_derog', 'mort_acc', 'is_bad']
Saved to: clean_loans_final.csv


In [ ]:
NamesOfCols = []
NamesOfCols = df_clean.columns

for i in NamesOfCols:
    print(i)

In [ ]:
# Check missing values

missing = df_clean.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

In [ ]:
# 5. Save the cleaned version
df.to_csv(, index=False)
print("Saved to: clean_loans_final.csv")

In [ ]:
# 2. CREATE TARGET (1 = Default, 0 = Good)
bad = ['Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off']
df['is_bad'] = np.where(df['loan_status'].isin(bad), 1, 0)

# Remove loans still active (unknown outcome)
df = df[~df['loan_status'].isin(['Current', 'In Grace Period'])]
print(f"Remaining: {len(df)} | Default rate: {df['is_bad'].mean()*100:.2f}%")